<a href="https://colab.research.google.com/github/mp3pintyo/googlecolab/blob/main/HeartLib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Setup Environment
import os

!sudo apt-get update && sudo apt-get install -y ffmpeg

if not os.path.exists("heartlib"):
    !git clone https://github.com/HeartMuLa/heartlib.git

%cd heartlib

!pip install -e .
!pip install -u "huggingface_hub[cli]"
!pip install flash-attn --no-build-isolation
!pip install accelerate

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,312 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,650 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-secu

In [2]:
# @title Download Checkpoints
import os

if os.path.basename(os.getcwd()) != "heartlib":
    %cd heartlib

os.makedirs('./ckpt', exist_ok=True)

!huggingface-cli download --local-dir './ckpt' 'HeartMuLa/HeartMuLaGen'
!huggingface-cli download --local-dir './ckpt/HeartMuLa-oss-3B' 'HeartMuLa/HeartMuLa-oss-3B'
!huggingface-cli download --local-dir './ckpt/HeartCodec-oss' 'HeartMuLa/HeartCodec-oss'
!huggingface-cli download --local-dir './ckpt/HeartTranscriptor-oss' 'HeartMuLa/HeartTranscriptor-oss'

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]Downloading 'gen_config.json' to 'ckpt/.cache/huggingface/download/EpjBFd09FpMaURtnmYyhqAvQM-s=.3a5413112a5917ce99497eb3ba9ac97072fb30f2.incomplete'

gen_config.json: 100% 101/101 [00:00<00:00, 946kB/s]
Download complete. Moving file to ckpt/gen_config.json

README.md: 1.89kB [00:00, 10.6MB/s]
Download complete. Moving file to ckpt/README.md

tokenizer.json: 0.00B [00:00, ?B/s]Downloading '.gitattributes' to 'ckpt/.cache/huggingface/download/wPaCkH-WbT7GsmxMKKrNZTV4nSM=.a6344aac8c09253b3b630fb776ae94478aa0275b.incomplete'


.gitattributes: 1.52kB [00:00, 8.87MB/s]
Download complete. Moving file to ckpt/.gitattributes
tokenizer.json: 9.09MB [00:00, 126MB/s]
Download complete. Moving file to ckpt/tokenizer.json
Fetching 4 files: 100% 4/4 [00:00<00:00, 16.71it/s]
/content/heartlib/ckpt
⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.

In [4]:
# @title Run Music Generation
# No Touchy
import os
import tempfile
from IPython.display import Audio, display
import torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

# -----------------------------------------------------------------------------
# 1. SETUP YOUR LYRICS AND TAGS (you can touch this lol)
# -----------------------------------------------------------------------------
my_lyrics = """
[Verse]
The sun creeps in across the floor
I hear the traffic outside the door
The coffee pot begins to hiss
It is another morning just like this

[Chorus]
Every day the light returns
Every day the fire burns
"""

my_tags = "piano,happy,pop"
# -----------------------------------------------------------------------------
# Do Not Touch This Code Below, lol.

if os.path.basename(os.getcwd()) != "heartlib":
    if os.path.exists("heartlib"):
        %cd heartlib
    else:
        raise FileNotFoundError("Repo not found. Please run Block 1 (Setup) first.")

# Write lyrics/tags to temp files to avoid shell quoting issues
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as lyrics_file:
    lyrics_file.write(my_lyrics)
    lyrics_path = lyrics_file.name

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8") as tags_file:
    tags_file.write(my_tags)
    tags_path = tags_file.name

!python ./examples/run_music_generation.py \
    --model_path=./ckpt \
    --version="3B" \
    --lyrics="{lyrics_path}" \
    --tags="{tags_path}" \
    --save_path="./assets/output.mp3" \
    --lazy_load=true

os.unlink(lyrics_path)
os.unlink(tags_path)

if os.path.exists("./assets/output.mp3"):
    print("✓ Generation complete!")
    display(Audio("./assets/output.mp3"))
else:
    print("✗ Generation failed. Check logs above.")


2026-01-26 23:41:22.966751: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769470883.006824    7642 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769470883.019924    7642 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769470883.065296    7642 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769470883.065339    7642 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769470883.065346    7642 computation_placer.cc:177] computation placer alr